**v5** (2026-09-19) — GPU JPEG decode, lazy real frames, mmap loads (RAM fix)

# Visual-inertial encoder training (Colab)

Trains the triage visual encoder on pre-collected rollout data.
The Rust sim is not needed — data is uploaded via Drive.

**Before running:**
1. Runtime → Change runtime type → GPU (T4 or better)
2. Upload to `My Drive/triage-data/`:
   - `train-256-rollouts.pt`
   - `val-256-rollouts.pt`
   - `real-256.pt`
3. Run all cells top to bottom.


In [ ]:
!nvidia-smi -L
import torch
print(torch.__version__, torch.cuda.is_available())

## 1. Get the code

Fetches the needed `.py` files from GitHub raw at a pinned commit —
no git clone, no auth, no stale caches.


In [ ]:
import os, urllib.request
from pathlib import Path

SHA = "22d3a86"  # bump to update code
BASE = f"https://raw.githubusercontent.com/jarenm1/triage/{SHA}"

FILES = [
    "rl/__init__.py",
    "rl/visual_encoder.py",
    "rl/venc_augment.py",
    "rl/venc_domain.py",
    "rl/venc_train.py",
    "rl/venc_eval.py",
    "rl/vendor/__init__.py",
    "rl/vendor/lingbot_vision/__init__.py",
    "rl/vendor/lingbot_vision/attention.py",
    "rl/vendor/lingbot_vision/build.py",
    "rl/vendor/lingbot_vision/layers.py",
    "rl/vendor/lingbot_vision/loader.py",
    "rl/vendor/lingbot_vision/preprocess.py",
    "rl/vendor/lingbot_vision/vit.py",
    "rl/vendor/lingbot_vision/configs/lbot_vision_vitb.yaml",
]

os.chdir("/content")
for f in FILES:
    dest = Path("triage") / f
    dest.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{BASE}/{f}", dest)
    print("fetched", f)

os.chdir("/content/triage")
print("cwd:", os.getcwd())


In [ ]:
# (no alternative needed — cell above fetches everything)


## 2. Install deps

In [ ]:
!pip install -q omegaconf huggingface_hub
import sys
sys.path.insert(0, ".")
from rl.visual_encoder import VisualEncoder, VisualEncoderConfig
enc = VisualEncoder()
print("encoder ok, params:", sum(p.numel() for p in enc.parameters()))

## 3. Upload data

Three files, ~5GB total — Drive required. Frames are JPEG-packed
(decode on batch in the trainer).

- `train-256-rollouts.pt` — sim rollouts, 256×192, paired appearances
- `val-256-rollouts.pt` — held-out sim rollouts
- `real-256.pt` — 23.5k real FPV frames (19 clips)


In [ ]:
import os
from pathlib import Path

DATA = Path("target/venc-data")
DATA.mkdir(parents=True, exist_ok=True)

from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/triage-data/train-256-rollouts.pt target/venc-data/
!cp /content/drive/MyDrive/triage-data/val-256-rollouts.pt target/venc-data/
!cp /content/drive/MyDrive/triage-data/real-256.pt target/venc-data/

for p in [DATA/"train-256-rollouts.pt", DATA/"val-256-rollouts.pt", DATA/"real-256.pt"]:
    print(p, "exists:", p.exists())


## 4. Train

Variants: `a`–`h` (core ablation), `i` (+appearance consistency), `j` (+DINOv2 distill),
`k` (+domain adversarial on real frames), `l` (k + teacher distill).

Teacher options: `dinov2` (open), `lingbot` (ViT-L/16 dense-perception, open), `dinov3` (gated, needs `--teacher-ckpt`).

Recommended: variant `l` with `--teacher lingbot` — the full sim↔real alignment stack.

In [ ]:
import subprocess
VARIANT = "m"          # distill on sim+real, disc eval-only
TEACHER = "lingbot"
TEACHER_VARIANT = "base"   # ViT-B: fits T4 alongside the student
STEPS = 4000
BATCH = 16

cmd = (
    f"python -m rl.venc_train --variant {VARIANT} "
    f"--data target/venc-data/train-256-rollouts.pt "
    f"--val target/venc-data/val-256-rollouts.pt "
    f"--real target/venc-data/real-256.pt "
    f"--teacher {TEACHER} --teacher-variant {TEACHER_VARIANT} "
    f"--batch {BATCH} --steps {STEPS} "
    f"--output target/venc-results/{VARIANT}-256-colab.json"
)
print(cmd, flush=True)
# streams stdout/stderr live to the cell; prints exit code at end
r = subprocess.run(cmd, shell=True, cwd="/content/triage")
print("exit:", r.returncode)


## 5. Results

In [ ]:
import json
r = json.load(open(f"target/venc-results/{VARIANT}-colab.json"))
print(json.dumps({"bench": r["bench"], "final": r["final"]}, indent=2))

In [ ]:
# Occlusion retention eval
import subprocess
r = subprocess.run(
    f"python -m rl.venc_eval "
    f"--checkpoint target/venc-results/{VARIANT}-256-colab.pt "
    f"--variant {VARIANT} --val target/venc-data/val-256-rollouts.pt",
    shell=True, cwd="/content/triage")
print("exit:", r.returncode)


In [ ]:
# Download results + checkpoint
# from google.colab import files
# files.download(f"target/venc-results/{VARIANT}-colab.json")
# files.download(f"target/venc-results/{VARIANT}-colab.pt")